# Лабораторная 02. Partitions, tasks и parallelism

Цель: увидеть связь `partitions -> tasks -> parallelism`.

In [4]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder.appName('lab-02-partitions').master('local[*]')
    .config('spark.driver.memory', '2g')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.sql.adaptive.enabled', 'false')
    .getOrCreate())
spark.sparkContext.setLogLevel('WARN')


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/01 16:07:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
from pathlib import Path
base_uri = Path('spark_core_data').absolute().as_uri()
orders = spark.read.parquet(f'{base_uri}/orders')
print('Spark UI:', spark.sparkContext.uiWebUrl)

Spark UI: http://3bbcc89f06d1:4040


## Задание 1
Посмотрите исходное количество partitions.

In [6]:
orders.rdd.getNumPartitions()

4

## Задание 2
Уменьшите partitions через `coalesce(2)` и увеличьте через `repartition(20)`.

In [7]:
orders_2 = orders.coalesce(2)
orders_20 = orders.repartition(20)
print('orders:', orders.rdd.getNumPartitions())
print('orders_2:', orders_2.rdd.getNumPartitions())
print('orders_20:', orders_20.rdd.getNumPartitions())

orders: 4
orders_2: 2
orders_20: 20


## Задание 3
Для каждого DataFrame вызовите `count()`. После каждого action смотрите количество tasks в Spark UI на вкладке Stages.

In [8]:
orders.count()

120000

In [9]:
orders_2.count()

120000

In [10]:
orders_20.count()

[Stage 6:==========================================>              (15 + 4) / 20]

120000

Заполните таблицу:

| DataFrame | Num partitions | Action | Tasks в Spark UI |
|---|---:|---|---:|
| orders | 4 | count() | 5 |
| orders_2 | 2 | count() | 3 |
| orders_20 | 20 | count() | 25 |

Вопросы:

- Почему `coalesce(2)` уменьшил parallelism?
  потому что уменьшилось число партиций
- Почему `repartition(20)` может добавить shuffle?
  чтобы перемешать данные из 4 исходных партиций и распределить по 20 новым
- Когда много partitions становится overhead?
  когда спарк тратит слишком много времени на планировку, слишком много метаданных о каждой маленькой задачке

In [11]:
spark.stop()